# Week 01 – Python for Data Science: NumPy & Pandas Essentials

> **Thursday Learning Hours · Session 01**

### Learning Objectives
1. Create and manipulate NumPy arrays (indexing, slicing, broadcasting, vectorised operations).
2. Load, explore and clean tabular data with Pandas DataFrames.
3. Apply common data-wrangling patterns (filtering, groupby, merge, pivot).
4. Understand the performance differences between pure Python loops and NumPy/Pandas vectorisation.

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print(f'NumPy  version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

---
## Part 1 – NumPy Fundamentals

NumPy's `ndarray` is the workhorse of scientific Python. It stores homogeneous data in a contiguous memory block, enabling fast element-wise operations.

### 1.1 Array Creation

In [ ]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print('1-D array:', a, '| dtype:', a.dtype, '| shape:', a.shape)

# 2-D array (matrix)
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print('\n2-D array (matrix):')
print(M, '| shape:', M.shape)

# Built-in constructors
print('\nzeros(3,4):\n', np.zeros((3, 4)))
print('\nones(2,3) * 7:\n', np.ones((2, 3)) * 7)
print('\narange(0, 10, 2):', np.arange(0, 10, 2))
print('linspace(0, 1, 6): ', np.linspace(0, 1, 6))
print('\nIdentity (3x3):\n', np.eye(3))

# Random arrays
rng = np.random.default_rng(seed=42)
R = rng.standard_normal((3, 3))
print('\nRandom normal (3x3):\n', R.round(3))

### 1.2 Indexing & Slicing

In [ ]:
arr = np.arange(12).reshape(3, 4)
print('Original array:\n', arr)

# Standard indexing
print('\nElement [1, 2]:', arr[1, 2])
print('Row 0:         ', arr[0])
print('Column 2:      ', arr[:, 2])
print('Sub-matrix [0:2, 1:3]:\n', arr[0:2, 1:3])

# Boolean indexing
mask = arr > 6
print('\nElements > 6:', arr[mask])

# Fancy indexing
print('Rows [0, 2]:\n', arr[[0, 2]])

### 1.3 Mathematical Operations & Universal Functions (ufuncs)

In [ ]:
x = np.array([1.0, 4.0, 9.0, 16.0])

print('x          :', x)
print('sqrt(x)    :', np.sqrt(x))
print('log(x)     :', np.log(x).round(3))
print('exp(x)     :', np.exp(x))
print('x ** 2     :', x ** 2)
print('x + 10     :', x + 10)

# Aggregate functions
A = rng.integers(1, 20, size=(4, 5))
print('\nRandom integer matrix:\n', A)
print('Sum (all)   :', A.sum())
print('Sum (cols)  :', A.sum(axis=0))   # along rows → per column
print('Sum (rows)  :', A.sum(axis=1))   # along cols → per row
print('Mean        :', A.mean().round(3))
print('Std         :', A.std().round(3))
print('Min / Max   :', A.min(), '/', A.max())

### 1.4 Broadcasting

Broadcasting lets NumPy perform arithmetic on arrays of **different shapes** by automatically expanding dimensions. Understanding the rules avoids bugs and unlocks elegant, loop-free code.

In [ ]:
# Example 1: scalar ↔ array
v = np.array([10, 20, 30])
print('v + 5 =', v + 5)          # scalar broadcast to shape (3,)

# Example 2: (3,1) ↔ (3,)
col = np.array([[1], [2], [3]])   # shape (3, 1)
row = np.array([10, 20, 30])      # shape (3,)  → treated as (1, 3)
print('\ncol + row (broadcasting):\n', col + row)

# Example 3: Z-score normalisation without a loop
data = rng.standard_normal((100, 5))   # 100 samples, 5 features
z_scores = (data - data.mean(axis=0)) / data.std(axis=0)
print('\nZ-score normalised data (first 3 rows):\n', z_scores[:3].round(3))
print('Column means (should be ≈0):', z_scores.mean(axis=0).round(10))
print('Column stds  (should be ≈1):', z_scores.std(axis=0).round(10))

### 1.5 Vectorisation Speed Comparison

Let's time a dot product computed three ways: a pure Python loop, NumPy dot, and NumPy matmul.

In [ ]:
N = 1_000_000
a = rng.standard_normal(N)
b = rng.standard_normal(N)

# Pure Python loop
t0 = time.perf_counter()
result_loop = sum(ai * bi for ai, bi in zip(a, b))
t_loop = time.perf_counter() - t0

# NumPy dot
t0 = time.perf_counter()
result_np = np.dot(a, b)
t_np = time.perf_counter() - t0

print(f'Python loop : {t_loop*1000:.1f} ms  |  result = {result_loop:.6f}')
print(f'NumPy dot   : {t_np*1000:.3f} ms  |  result = {result_np:.6f}')
print(f'Speedup     : {t_loop/t_np:.0f}×')

---
## Part 2 – Pandas Fundamentals

Pandas introduces two labelled data structures built on NumPy arrays:
- **`Series`** – 1-D array with an index.
- **`DataFrame`** – 2-D table of `Series` sharing a common index.

### 2.1 Series

In [ ]:
# Create a Series from a dict
populations = pd.Series(
    {'California': 39_538_223,
     'Texas':      29_145_505,
     'Florida':    21_538_187,
     'New York':   20_201_249,
     'Illinois':   12_812_508},
    name='Population (2020 Census)'
)
print(populations)
print('\nIndex  :', populations.index.tolist())
print('Values :', populations.values)
print('\nTexas  :', populations['Texas'])
print('\nStates with pop > 25M:\n', populations[populations > 25_000_000])

### 2.2 Creating & Inspecting a DataFrame

In [ ]:
# Synthetic student dataset
rng2 = np.random.default_rng(seed=7)

n = 200
df = pd.DataFrame({
    'student_id': [f'S{i:04d}' for i in range(1, n + 1)],
    'age':        rng2.integers(18, 30, size=n),
    'gender':     rng2.choice(['M', 'F', 'Other'], size=n, p=[0.48, 0.48, 0.04]),
    'department': rng2.choice(['CS', 'EE', 'ME', 'CE', 'DS'], size=n),
    'gpa':        np.clip(rng2.normal(loc=3.1, scale=0.5, size=n), 0, 4.0).round(2),
    'hours_study':rng2.integers(1, 12, size=n),
    'passed':     rng2.choice([True, False], size=n, p=[0.75, 0.25]),
})

# Introduce some missing values intentionally
missing_idx = rng2.choice(df.index, size=10, replace=False)
df.loc[missing_idx, 'gpa'] = np.nan

print('Shape:', df.shape)
print()
df.head(8)

In [ ]:
# Quick inspection methods
print('--- df.info() ---')
df.info()
print()
print('--- df.describe() ---')
df.describe().round(2)

### 2.3 Selection – `[]`, `.loc`, `.iloc`

In [ ]:
# Single column → Series
print('GPA Series (first 5):'); print(df['gpa'].head()); print()

# Multiple columns → DataFrame
print('student_id & gpa (first 5):'); print(df[['student_id', 'gpa']].head()); print()

# Label-based (.loc)  – row label : column name
print('.loc[0:3, ["age","department","gpa"]]:')
print(df.loc[0:3, ['age', 'department', 'gpa']]); print()

# Integer-based (.iloc) – row int : col int
print('.iloc[0:3, 1:4]:')
print(df.iloc[0:3, 1:4]); print()

# Boolean filtering
cs_high_gpa = df[(df['department'] == 'CS') & (df['gpa'] > 3.5)]
print(f'CS students with GPA > 3.5: {len(cs_high_gpa)} students')
cs_high_gpa[['student_id', 'department', 'gpa', 'passed']].head()

### 2.4 Handling Missing Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nRows with any NaN: {df.isnull().any(axis=1).sum()}')

# Strategy 1: drop rows with NaN
df_dropped = df.dropna()
print(f'\nRows after dropna: {len(df_dropped)}')

# Strategy 2: fill NaN with column mean
df_filled = df.copy()
df_filled['gpa'] = df_filled['gpa'].fillna(df_filled['gpa'].mean())
print(f'Missing GPA after fillna: {df_filled["gpa"].isnull().sum()}')

### 2.5 GroupBy & Aggregation

In [ ]:
dept_stats = (
    df_filled
    .groupby('department')
    .agg(
        count       =('student_id', 'count'),
        avg_gpa     =('gpa',         'mean'),
        avg_hours   =('hours_study', 'mean'),
        pass_rate   =('passed',       'mean'),
    )
    .round(3)
    .sort_values('avg_gpa', ascending=False)
)
print('Department statistics:')
dept_stats

### 2.6 Merge / Join

In [ ]:
# Create a supplementary 'departments' lookup table
dept_info = pd.DataFrame({
    'department':  ['CS',              'EE',                'ME',                'CE',              'DS'],
    'full_name':   ['Computer Science','Electrical Eng.',   'Mechanical Eng.',   'Civil Eng.',      'Data Science'],
    'founded_year':[1960,              1955,                1940,                1935,              2010],
})

df_merged = df_filled.merge(dept_info, on='department', how='left')
print('Merged DataFrame (first 5 rows):')
df_merged[['student_id', 'department', 'full_name', 'gpa']].head()

### 2.7 Pivot Table

In [ ]:
pivot = df_filled.pivot_table(
    values='gpa',
    index='department',
    columns='gender',
    aggfunc='mean'
).round(3)
print('Average GPA by Department × Gender:')
pivot

---
## Part 3 – Quick Visualisation

Pandas has built-in plotting backed by Matplotlib. A picture is worth a thousand rows.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Week 01 – Student Dataset Quick EDA', fontsize=13)

# GPA distribution
axes[0].hist(df_filled['gpa'].dropna(), bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('GPA Distribution')
axes[0].set_xlabel('GPA')
axes[0].set_ylabel('Count')

# Average GPA per department
dept_avg = df_filled.groupby('department')['gpa'].mean().sort_values(ascending=False)
axes[1].bar(dept_avg.index, dept_avg.values, color='coral', edgecolor='white')
axes[1].set_title('Avg GPA by Department')
axes[1].set_xlabel('Department')
axes[1].set_ylabel('Avg GPA')
axes[1].set_ylim(2.8, 3.5)

# Study hours vs GPA scatter
colours = df_filled['passed'].map({True: 'green', False: 'red'})
axes[2].scatter(df_filled['hours_study'], df_filled['gpa'], c=colours, alpha=0.5, s=20)
axes[2].set_title('Study Hours vs GPA\n(green=passed, red=failed)')
axes[2].set_xlabel('Study Hours / day')
axes[2].set_ylabel('GPA')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to eda_plots.png')

---
## Summary & Key Takeaways

| Concept | What we learned |
|---------|----------------|
| **ndarray** | Core N-D array; homogeneous, contiguous memory |
| **Indexing / Slicing** | `arr[row, col]`, boolean masks, fancy indexing |
| **Broadcasting** | Element-wise ops across mismatched shapes without loops |
| **Vectorisation** | NumPy ufuncs beat Python loops by 100–1000× |
| **Series / DataFrame** | Labelled 1-D & 2-D structures; `.loc` / `.iloc` |
| **Missing values** | `dropna()` / `fillna()` strategies |
| **GroupBy** | Split-apply-combine: `groupby().agg()` |
| **Merge** | SQL-style joins with `DataFrame.merge()` |
| **Pivot table** | Cross-tabulated aggregation with `pivot_table()` |

**Next session (Week 02):** Data Visualization with Matplotlib & Seaborn — going beyond simple plots to publication-quality charts.